This notebook is intended to assist with performing pilot studies to determine the correct numer of trials required to get a statistically-meaningful result when running benchmarks. It assumes you have already run (and tagged) samples. It will tehn look at them to determine how large your sample size needs to be to determine if an effect exists.

We begin by setting some variables:

In [ ]:
# # The number of samples to run in our pilot study. More samples here MAY result in a lower number of tests required in the actual experiment.
# TRIAL_SAMPLE=20

# α (Alpha) is the boundry for how likely it is that you will have a false positive (find an effect when none exists)
ALPHA=0.05

# Power is roughly the chance that you detect an effect, if one exists.
POWER=0.8

# The query for your baseline case (we assume everything is being compared to this test case)
baseline_query="test==\"no_monitoring\" and (\"jobsize\" not in locals() or jobsize==\"E\")"

# The queries for your test cases - the key will be used as the name.
experimental_queries={ 
    "HPCPerfStats": "test==\"hpcperfstats\"  and (\"jobsize\" not in locals() or jobsize==\"E\")",
    "LDMS": "test==\"ldms\" and (\"jobsize\" not in locals() or jobsize==\"E\")",
    "BMC": "test==\"bmc_with_amsd_clean\" and (\"jobsize\" not in locals() or jobsize==\"E\")",
}

# If needed, this can be used to limit the date range
date_range = "" # for example: now-1d:now

# If needed, any extra arguments for reframe
reframe_extra_args = ""

Next we grab data about the studies and transform it to an easy to work with data structure:

In [ ]:
from json import loads

json={}
tagsets = { "baseline": baseline_query } | experimental_queries

for key in tagsets:
    # Call reframe to get the JSON representation of all of our baseline jobs.
    jres = ! ./reframe.sh {reframe_extra_args} --describe-stored-sessions '{date_range}?{tagsets[key]}'
    json[key] = loads('\n'.join(jres)) # Join it into one string

from pprint import pprint
pprint(json)

import re

perf = {} # Scenario, then name of test, then performance metric, containing a list of test results
metric_name_re = re.compile(r'[^:]+:[^:]+:([^:]+)')

for scenario in json:
    perf[scenario] = {}
    for session in json[scenario]:
        for run in session['runs']:
            for testcase in run['testcases']:
                if testcase['fail_phase'] is None:
                    testname = testcase['name']
                    if testname not in perf[scenario]:
                        perf[scenario][testname] = {}
                    for metric in testcase['perfvalues']:
                        # Do some string processing to get the name of the perf counter
                        metric_name = metric_name_re.match(metric).group(1)
                        # Check if that counter already exists and add if not
                        if metric_name not in perf[scenario][testname]:
                            perf[scenario][testname][metric_name] = {
                                "values": [],
                                "unit": testcase['perfvalues'][metric][4]
                            }
                        perf[scenario][testname][metric_name]['values'].append(
                            testcase['perfvalues'][metric][0]
                        )
pprint(perf)

And finally, for each metric of each experimental scenario, we perform the power test to determine how many tests are necessary.

In [ ]:
from math import sqrt, ceil
from numpy import var, mean, ndarray, array as nparr
from statsmodels.stats.power import TTestIndPower

scenarios = list(perf.keys())
scenarios.remove('baseline')

for scenario in scenarios:
    print(perf['baseline'].keys())
    for test in perf['baseline'].keys():
        for metric in perf['baseline'][test].keys():
            basedata = perf['baseline'][test][metric]['values']
            exprdata = perf[scenario][test][metric]['values']

            lb = len(basedata)
            le = len(exprdata)

            vb = var(basedata)
            ve = var(exprdata)

            mb = mean(basedata)
            me = mean(exprdata)

            pooled_std = sqrt(((lb - 1) * vb) + (le - 1) * ve) / ( lb + le - 2)

            effect_size = (mb - me) / pooled_std

            res = TTestIndPower().solve_power(
                effect_size=effect_size, alpha=ALPHA, power=POWER, ratio=1,
                alternative='two-sided'
            )
            if isinstance(res, ndarray):
                print(f"{scenario}: {test}: {metric}: Pilot study may already have achieved significance...")
            else:
                print(f"""{scenario}: {test}: {metric}: 
    Effect size: {effect_size}
    Required samples: {ceil(res)}""")
                TTestIndPower().plot_power(
                    nobs=nparr(range(10,2*ceil(res),10)), alpha=ALPHA,
                    effect_size=[effect_size]
                )